# NOTEBOOK 04: PYTORCH DEEP LEARNING (MLP NEURAL NETWORK)
## Architecture
- Input: 8 features
- Layer 1: Linear(8, 64) + BatchNorm1d + ReLU + Dropout(0.2)
- Layer 2: Linear(64, 32) + BatchNorm1d + ReLU + Dropout(0.2)
- Layer 3: Linear(32, 1) Logits
- Loss: `BCEWithLogitsLoss` with `pos_weight = 10.76` (inverse class prevalence)


In [1]:
import os
import sys
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

sys.path.append(os.path.join("..", "training_scripts"))
from data_loader import prepare_splits
from train_dl import DiabetesMLP

data = prepare_splits()
X_train, y_train = data['X_train'], data['y_train']
X_val, y_val = data['X_val'], data['y_val']
X_test, y_test = data['X_test'], data['y_test']

print(f"PyTorch version: {torch.__version__}")


PyTorch version: 2.14.0+cpu


### 4.1 PyTorch Model Definition and Initialization

In [2]:
model = DiabetesMLP(input_dim=8)
print(model)
pos_weight = torch.tensor([np.sum(y_train == 0) / (np.sum(y_train == 1) + 1e-5)])
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
print(f"Loss weighted for positive class with factor: {pos_weight.item():.2f}")


DiabetesMLP(
  (net): Sequential(
    (0): Linear(in_features=8, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.2, inplace=False)
    (8): Linear(in_features=32, out_features=1, bias=True)
  )
)


Loss weighted for positive class with factor: 10.76


### 4.2 Training Loop with Validation Monitoring

In [3]:
train_ds = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32).unsqueeze(1))
val_ds = TensorDataset(torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.float32).unsqueeze(1))
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False)

for epoch in range(1, 6):
    model.train()
    for bx, by in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(bx), by)
        loss.backward()
        optimizer.step()
    
    model.eval()
    val_preds, val_targets = [], []
    with torch.no_grad():
        for bx, by in val_loader:
            probs = torch.sigmoid(model(bx)).numpy().flatten()
            val_preds.extend((probs >= 0.5).astype(int))
            val_targets.extend(by.numpy().flatten())
    print(f"Epoch {epoch:02d} | Val F1: {f1_score(val_targets, val_preds):.4f} | Recall: {recall_score(val_targets, val_preds):.4f}")


Epoch 01 | Val F1: 0.5454 | Recall: 0.8976


Epoch 02 | Val F1: 0.5639 | Recall: 0.8824


Epoch 03 | Val F1: 0.5688 | Recall: 0.8894


Epoch 04 | Val F1: 0.5772 | Recall: 0.8906


Epoch 05 | Val F1: 0.5784 | Recall: 0.9047
